In [1]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
import itertools

**DESCRIPTORS**

In [2]:
def sift_descriptor(
    img: np.ndarray,
    params
):
    if params is None:
        params = {}

    sift = cv2.SIFT_create(
        nfeatures=params.get("nfeatures", 0),
        nOctaveLayers=params.get("nOctaveLayers", 3),
        contrastThreshold=params.get("contrastThreshold", 0.04),
        edgeThreshold=params.get("edgeThreshold", 10),
        sigma=params.get("sigma", 1.6),
    )

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    keypoints, descriptors = sift.detectAndCompute(gray, None)
    return keypoints, descriptors


def orb_descriptor(
    img: np.ndarray,
    params
):
    if params is None:
        params = {}

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img

    orb = cv2.ORB_create(
        nfeatures=params.get("nfeatures", 500),
        scaleFactor=params.get("scaleFactor", 1.2),
        nlevels=params.get("nlevels", 8),
        edgeThreshold=params.get("edgeThreshold", 31),
        firstLevel=params.get("firstLevel", 0),
        WTA_K=params.get("WTA_K", 2),
        scoreType=params.get("scoreType", cv2.ORB_HARRIS_SCORE),
        patchSize=params.get("patchSize", 31),
        fastThreshold=params.get("fastThreshold", 20),
    )

    keypoints, descriptors = orb.detectAndCompute(gray, None)
    return keypoints, descriptors


def color_sift_descriptor(
    img: np.ndarray,
    params
):
    if params is None:
        params = {}

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    sift = cv2.SIFT_create(
        nfeatures=params.get("nfeatures", 0),
        nOctaveLayers=params.get("nOctaveLayers", 3),
        contrastThreshold=params.get("contrastThreshold", 0.04),
        edgeThreshold=params.get("edgeThreshold", 10),
        sigma=params.get("sigma", 1.6),
    )

    keypoints = sift.detect(v, None)

    desc_h = np.zeros((len(keypoints), 128))
    desc_s = np.zeros_like(desc_h)
    desc_v = np.zeros_like(desc_h)

    if keypoints:
        _, desc_h = sift.compute(h, keypoints)
        _, desc_s = sift.compute(s, keypoints)
        _, desc_v = sift.compute(v, keypoints)

    descriptors = np.concatenate((desc_h, desc_s, desc_v), axis=1)
    return keypoints, descriptors


**PRECOMPUTATION DB DESCRIPTORS USING DIFFERENT HYPERPARAMETERS**

In [3]:
bbdd_path = "../data/BBDD"
current_dir = os.getcwd()
output_root = os.path.join(current_dir, "BBDD_DESCRIPTORS")
os.makedirs(output_root, exist_ok=True)

descriptor_funcs = {
    "sift": sift_descriptor,
    "orb": orb_descriptor,
    "color_sift": color_sift_descriptor
}

param_grids = {
    "sift": {
        "sigma": [1.2, 1.6],
        "edgeThreshold": [6,8, 10],
        "nOctaveLayers": [3, 5],
    },
    "orb": {
        "nfeatures": [500, 1000],
        "fastThreshold": [10, 20],
    },
    "color_sift": {
        "sigma": [1.4, 1.6],
        "edgeThreshold": [6,8, 10],
        "nOctaveLayers": [3, 5],
    },
}

image_files = [f for f in os.listdir(bbdd_path) if f.lower().endswith((".jpg"))]

for descriptor, func in descriptor_funcs.items():
    print(f"Processing: {descriptor.upper()}")

    param_grid = param_grids.get(descriptor, {})

    if not param_grid:
        param_combinations = [{}]
    else:
        keys, values = zip(*param_grid.items())
        param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

    for params in param_combinations:

        param_name = "_".join([f"{k}{v}" for k, v in params.items()]) if params else "default"
        folder_name = f"BBDD_{descriptor}_{param_name}"
        output_dir = os.path.join(output_root, folder_name)
        os.makedirs(output_dir, exist_ok=True)

        print(f"\n{descriptor.upper()} with {params}")

        for img_name in image_files:
            img_path = os.path.join(bbdd_path, img_name)
            img = cv2.imread(img_path)

            keypoints, descriptors = func(img, params=params)
            coords = np.array([kp.pt for kp in keypoints], dtype=np.float32)

            base_name = os.path.splitext(img_name)[0]
            save_path = os.path.join(output_dir, f"{base_name}_{descriptor}.npz")
            np.savez_compressed(save_path, keypoints=coords, descriptors=descriptors)

        print(f"Saved in {output_dir}")


Processing: SIFT

SIFT with {'sigma': 1.2, 'edgeThreshold': 6, 'nOctaveLayers': 3}
Saved in c:\Users\xavipba\OneDrive\Escritorio\Msc Computer vision\C1 Project\Team3\exploratory_analysis\BBDD_DESCRIPTORS\BBDD_sift_sigma1.2_edgeThreshold6_nOctaveLayers3

SIFT with {'sigma': 1.2, 'edgeThreshold': 6, 'nOctaveLayers': 5}
Saved in c:\Users\xavipba\OneDrive\Escritorio\Msc Computer vision\C1 Project\Team3\exploratory_analysis\BBDD_DESCRIPTORS\BBDD_sift_sigma1.2_edgeThreshold6_nOctaveLayers5

SIFT with {'sigma': 1.2, 'edgeThreshold': 8, 'nOctaveLayers': 3}
Saved in c:\Users\xavipba\OneDrive\Escritorio\Msc Computer vision\C1 Project\Team3\exploratory_analysis\BBDD_DESCRIPTORS\BBDD_sift_sigma1.2_edgeThreshold8_nOctaveLayers3

SIFT with {'sigma': 1.2, 'edgeThreshold': 8, 'nOctaveLayers': 5}
Saved in c:\Users\xavipba\OneDrive\Escritorio\Msc Computer vision\C1 Project\Team3\exploratory_analysis\BBDD_DESCRIPTORS\BBDD_sift_sigma1.2_edgeThreshold8_nOctaveLayers5

SIFT with {'sigma': 1.2, 'edgeThreshol

FOR EACH DESCRIPTOR COMBINATION, GENERATE THE DEVELOPMENT SET DESCRIPTOR, SELECT THE PRECOMPUTED DB DESCRIPTORS, HYPERPARAMETER SEARCH USING DIFFERENT SIMILARITY METRICS / FILTERING OF KEYPOINTS / FINAL SCORE

In [ ]:
import json
from utils import global_metrics
from utils.local_metrics import sift_match_count, sift_match_normalized, sift_match_geometric

bbdd_path = "../data/BBDD"
dev_path = "../data/qsd1_w4"
output_root = os.path.join(os.getcwd(), "BBDD_DESCRIPTORS")

descriptor_funcs = {
    "sift": sift_descriptor,
    "orb": orb_descriptor,
    "color_sift": color_sift_descriptor
}

WANTED_DISTANCES = [
    global_metrics.euclidean_distance,
    global_metrics.x2_dist,
    global_metrics.bhattacharyya_distance,
    global_metrics.l1_distance,
    (global_metrics.histogram_intersection, 1),
    (global_metrics.hellinger_kernel, 1),
    global_metrics.earth_movers_distance,
    global_metrics.canberra_distance,
]

LOCAL_DISTANCES = [
    sift_match_count,
    sift_match_normalized,
    sift_match_geometric
]

def load_descriptors(folder):
    desc_data = {}
    for file in os.listdir(folder):
        if file.endswith(".npz"):
            base = os.path.splitext(file)[0]
            data = np.load(os.path.join(folder, file))
            desc_data[base] = {
                "keypoints": data["keypoints"],
                "descriptors": data["descriptors"]
            }
    return desc_data

def parse_params_from_name(param_str):
    params = {}
    if param_str == "default" or not param_str:
        return params
    for item in param_str.split("_"):
        key = ''.join([c for c in item if not c.isdigit() and c != '.'])
        val = ''.join([c for c in item if c.isdigit() or c == '.'])
        if val:
            params[key] = float(val) if '.' in val else int(val)
    return params


bbdd_descriptor_root = "../exploratory_analysis/BBDD_DESCRIPTORS"
dev_files = [f for f in os.listdir(dev_path) if f.lower().endswith(".jpg")]
results = {}

for folder in os.listdir(bbdd_descriptor_root):
    if not folder.startswith("BBDD_"):
        continue

    descriptor_type = folder.split("_")[1]
    param_str = "_".join(folder.split("_")[2:])
    print(f"Evaluating {descriptor_type.upper()} with {param_str}")

    bbdd_descriptors = load_descriptors(os.path.join(bbdd_descriptor_root, folder))
    func = descriptor_funcs[descriptor_type]
    params = parse_params_from_name(param_str)

    for dev_img in dev_files:
        dev_path_img = os.path.join(dev_path, dev_img)
        dev_img_data = cv2.imread(dev_path_img)
        dev_kp, dev_desc = func(dev_img_data, params=params)
        dev_coords = np.array([kp.pt for kp in dev_kp], dtype=np.float32)

        best_match = None
        best_score = float("inf")
        best_method = None

        for bbdd_name, bbdd_data in bbdd_descriptors.items():
            bbdd_desc = bbdd_data["descriptors"]
            bbdd_kp = bbdd_data["keypoints"]

            for dist_func in WANTED_DISTANCES:
                if isinstance(dist_func, tuple):
                    func_dist, weight = dist_func
                    d = 1 - func_dist(dev_desc, bbdd_desc)
                else:
                    d = dist_func(dev_desc, bbdd_desc)

                if d < best_score:
                    best_score = d
                    best_match = bbdd_name
                    best_method = str(dist_func)

            for local_func in LOCAL_DISTANCES:
                try:
                    sim = local_func(dev_kp=dev_coords, dev_desc=dev_desc,
                                     bbdd_kp=bbdd_kp, bbdd_desc=bbdd_desc)
                    d = 1 - sim 
                    if d < best_score:
                        best_score = d
                        best_match = bbdd_name
                        best_method = local_func.__name__
                except Exception:
                    continue

        results[dev_img] = {
            "best_match": best_match,
            "best_score": best_score,
            "method": best_method,
            "descriptor": descriptor_type,
            "params": params
        }
        print(f" - {dev_img}: best match {best_match} ({best_method}, score={best_score:.4f})")

with open("comparison_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("\n✅ Done! Results saved to comparison_results.json")
